# Machine Learning for Anti-Money Laundering

***
**Master's thesis: empirical part**

**Notebook 4: Results**

Benchmark: IBM AMLworld, variants HI-Small and LI-Small (Altman et al., 2023)
***

| Notebook | Content | Chapter 4 |
| --- | --- | --- |
| 01 Data and EDA | acquisition, integrity checks, temporal partition, EDA | 4.2–4.5 |
| 02 Features and feature selection | Sets A, B, C; causality verification; screening | 4.6 |
| 03 Models | one section per family: search, fit, threshold, evaluation | 4.7–4.8 |
| **04 Results** (this notebook) | comparison across families, sets and variants | 4.9 |

## Preliminaries

This notebook fits no model. It turns the records and scores stored by Notebook 03 into the
tables and figures of Chapter 5 and the statistical comparisons of Section 4.9: the rank-based
procedure of Demšar (2006) across families and feature sets, and a paired bootstrap of the test
block for the differences between feature sets. Every table is written to `results/tables/` and
every figure to `results/figures/`, with the prefix `nb04_`.

**Requirements:** Notebook 03 complete for both variants (150 final fits), same Google account.
CPU runtime. The bootstrap in Section 3 takes roughly 20–30 minutes; everything else, a few minutes.

In [ ]:
# Parameters
VARIANTS = ["HI-Small", "LI-Small"]
N_BOOT   = 1000      # bootstrap replicates for the feature-set contrasts (Section 3)
DRY_RUN  = False

In [ ]:
# Import relevant Python packages
import os, sys, json, platform, subprocess, datetime, warnings
from pathlib import Path

import numpy as np
import pandas as pd
from pandas import set_option
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import display
from sklearn.metrics import precision_recall_curve

set_option("display.precision", 4)
set_option("display.width", 160)
set_option("display.max_columns", 30)
sns.set_theme(context="notebook", style="whitegrid", palette="viridis")

In [ ]:
# Google Drive and project folder
try:
    from google.colab import drive
    drive.mount("/content/drive")
    BASE = Path("/content/drive/MyDrive")
except ImportError:
    BASE = Path.cwd()
PROJECT = BASE / ("aml-thesis-dryrun" if DRY_RUN else "aml-thesis")
assert (PROJECT / "results" / "models" / "final.jsonl").exists(), "run Notebook 03 first"
os.chdir(PROJECT)
if str(PROJECT) not in sys.path:
    sys.path.insert(0, str(PROJECT))
STORE = PROJECT / "results" / "models"
FIG, TAB = PROJECT / "results" / "figures", PROJECT / "results" / "tables"
def save_fig(fig, name):
    fig.savefig(FIG / f"nb04_{name}.png", dpi=200, bbox_inches="tight")
def save_tab(t, name):
    t.to_csv(TAB / f"nb04_{name}.csv")
print(f"project: {PROJECT}")

### Project code

In [ ]:
%%writefile aml/report.py
"""Notebook 04: tables, rank-based comparisons, paired bootstrap, typologies (Section 4.9).

Nothing here fits a model. Everything is computed from the records and scores that
Notebook 03 stored, together with the labels and typologies of the test rows.
"""
import json
from pathlib import Path

import numpy as np
import pandas as pd
from scipy import stats
from sklearn.metrics import average_precision_score

from . import config

FAMILY_NAMES = {"logreg": "Logistic regression", "svm": "Linear SVM", "rf": "Random forest",
                "hgb": "Gradient boosting", "ffnn": "Neural network"}
FAMILIES = list(FAMILY_NAMES)
SETS = ["A", "B", "C"]
# Critical values q_0.05 of the Nemenyi test (Demsar, 2006, Table 5), by number of treatments.
Q05 = {2: 1.960, 3: 2.343, 4: 2.569, 5: 2.728, 6: 2.850, 7: 2.949, 8: 3.031, 9: 3.102, 10: 3.164}


# --- records -------------------------------------------------------------------------------
def load_final(store_dir) -> pd.DataFrame:
    rows = []
    for line in (Path(store_dir) / "final.jsonl").read_text().splitlines():
        r = json.loads(line)
        t, v = r["test"], r["val"]
        rows.append({"variant": r["variant"], "family": r["family"], "set": r["set"],
                     "seed": r["seed"], "val_ap": v["ap"], "test_ap": t["ap"],
                     "test_roc_auc": t["roc_auc"], "recall_top": t["top_k"]["recall"],
                     "precision_top": t["top_k"]["precision"],
                     "recall_budget": t["at_budget"]["recall"],
                     "alert_rate_budget": t["at_budget"]["alert_rate"],
                     "precision_f1": t["at_f1"]["precision"], "recall_f1": t["at_f1"]["recall"],
                     "f1": t["at_f1"]["f1"], "alert_rate_f1": t["at_f1"]["alert_rate"],
                     "tau_f1": r["tau_f1"], "seconds": r["seconds"],
                     "n_pos": t["n_pos"], "n": t["n"],
                     "tail_tp": r["tail"]["at_f1"]["tp"] if r.get("tail") else 0,
                     "tail_fp": r["tail"]["at_f1"]["fp"] if r.get("tail") else 0,
                     "tail_fn": r["tail"]["at_f1"]["fn"] if r.get("tail") else 0,
                     "test_tp": t["at_f1"]["tp"], "test_fp": t["at_f1"]["fp"],
                     "test_fn": t["at_f1"]["fn"], "params": r["params"],
                     "extras": r.get("extras")})
    return pd.DataFrame(rows)


def mean_sd(df: pd.DataFrame, metric: str, digits: int = 3) -> pd.DataFrame:
    """'mean (sd)' strings, variant x family rows, set columns."""
    g = df.groupby(["variant", "family", "set"])[metric].agg(["mean", "std"])
    s = g["mean"].map(f"{{:.{digits}f}}".format) + " (" + g["std"].fillna(0).map(f"{{:.{digits}f}}".format) + ")"
    out = s.unstack("set")
    order = [(v, f) for v in out.index.get_level_values(0).unique() for f in FAMILIES
             if (v, f) in out.index]
    return out.loc[order]


# --- rank-based comparison (Demsar, 2006) ---------------------------------------------------
def friedman_nemenyi(table: pd.DataFrame) -> dict:
    """Rows are blocks (data conditions), columns are treatments; higher is better."""
    ranks = table.rank(axis=1, ascending=False)
    k, n = table.shape[1], table.shape[0]
    stat, p = stats.friedmanchisquare(*[table[c].to_numpy() for c in table.columns])
    cd = Q05[k] * np.sqrt(k * (k + 1) / (6.0 * n))
    return {"statistic": float(stat), "p_value": float(p), "n_blocks": n, "k": k,
            "average_ranks": ranks.mean().sort_values().to_dict(), "critical_difference": float(cd),
            "ranks": ranks}


# --- paired bootstrap of average precision ---------------------------------------------------
class SortedScores:
    """A score vector sorted once, so that average precision under any row weights is
    computed in one pass (the bootstrap re-weights rows instead of copying them). Only the
    thresholds at which a positive appears change recall, so precision is evaluated there."""

    def __init__(self, y, s):
        s = np.asarray(s, dtype=np.float64)
        self.order = np.argsort(-s, kind="mergesort")
        ss = s[self.order]
        ends = np.append(np.flatnonzero(np.diff(ss) != 0), len(ss) - 1)  # last row of each tie group
        self.pos_rows = np.flatnonzero(np.asarray(y, dtype=bool)[self.order])
        group_end = ends[np.searchsorted(ends, self.pos_rows)]
        self.last = np.append(np.flatnonzero(np.diff(group_end) != 0), len(group_end) - 1)
        self.group_end = group_end[self.last]

    def ap(self, w=None) -> float:
        """Average precision as in scikit-learn (step-wise, ties as one threshold)."""
        ws = np.ones(len(self.order)) if w is None else w[self.order]
        cum_all = np.cumsum(ws)[self.group_end]
        cum_tp = np.cumsum(ws[self.pos_rows])[self.last]
        if len(cum_tp) == 0 or cum_tp[-1] <= 0:
            return np.nan
        prec = np.divide(cum_tp, cum_all, out=np.zeros_like(cum_tp), where=cum_all > 0)
        d_rec = np.diff(np.concatenate([[0.0], cum_tp])) / cum_tp[-1]
        return float(np.sum(d_rec * prec))


def paired_bootstrap(y, scores: dict, contrasts: list, n_boot: int, seed: int = 0) -> pd.DataFrame:
    """scores[(group, set, seed)] -> test scores. For each contrast (group, high, low), the
    statistic is the seed-averaged difference AP(high) - AP(low). Every replicate draws one
    Poisson(1) weight per test row and applies it to all score vectors, so the comparison
    is paired across sets and seeds. Returns the observed difference and a percentile
    95 % interval of the bootstrap distribution."""
    y = np.asarray(y)
    prepared = {k: SortedScores(y, s) for k, s in scores.items()}
    seeds_of = {}
    for (g, st, sd) in scores:
        seeds_of.setdefault((g, st), []).append(sd)

    def stat(w):
        ap = {k: p.ap(w) for k, p in prepared.items()}
        out = {}
        for g, hi, lo in contrasts:
            common = sorted(set(seeds_of[(g, hi)]) & set(seeds_of[(g, lo)]))
            out[(g, hi, lo)] = np.mean([ap[(g, hi, s)] - ap[(g, lo, s)] for s in common])
        return out

    observed = stat(None)
    rng = np.random.default_rng(seed)
    draws = {c: np.empty(n_boot) for c in contrasts}
    for b in range(n_boot):
        w = rng.poisson(1.0, size=len(y)).astype(np.float64)
        for c, v in stat(w).items():
            draws[c][b] = v
    rows = []
    for (g, hi, lo) in contrasts:
        d = draws[(g, hi, lo)]
        lo_ci, hi_ci = np.nanpercentile(d, [2.5, 97.5])
        rows.append({"group": g, "contrast": f"{hi} - {lo}", "difference": observed[(g, hi, lo)],
                     "ci_low": lo_ci, "ci_high": hi_ci,
                     "share_of_replicates_above_0": float(np.nanmean(d > 0)),
                     "interval_excludes_0": bool(lo_ci > 0 or hi_ci < 0)})
    return pd.DataFrame(rows)


# --- labels, typologies and scores of the test rows ----------------------------------------
def test_rows(variant, features_dir=config.ROOT / "data" / "features",
              interim_dir=config.DATA_INTERIM) -> dict:
    """Labels and typologies in the row order used by Notebook 03 (test block without the
    tail, and the tail), with the alignment of the two files checked on file_row."""
    f = pd.read_parquet(Path(features_dir) / f"{variant}.parquet",
                        columns=["file_row", "block", "is_laundering", "tail"])
    i = pd.read_parquet(Path(interim_dir) / f"{variant}.parquet", columns=["file_row", "typology"])
    if not np.array_equal(f["file_row"].to_numpy(), i["file_row"].to_numpy()):
        raise RuntimeError("feature and interim rows are not aligned")
    block, tail = f["block"].astype(str).to_numpy(), f["tail"].to_numpy()
    test, tl = (block == "test") & ~tail, tail
    typ = i["typology"].cat.add_categories("no pattern").fillna("no pattern").astype(str).to_numpy()
    y = f["is_laundering"].to_numpy(np.int8)
    return {"y_test": y[test], "typology_test": typ[test], "y_tail": y[tl]}


def load_scores(store_dir, variant, family, set_name, seed):
    return np.load(Path(store_dir) / "scores" / f"{variant}_{family}_{set_name}_seed{seed}.npz")


def typology_recall(y, typology, scores_by_seed: dict, taus: dict, budget: float) -> pd.DataFrame:
    """Per typology: laundering transactions in the test block, share detected at the F1
    threshold and share among the highest-scoring `budget` share (both averaged over seeds)."""
    lab = y == 1
    k = max(1, int(np.ceil(budget * len(y))))
    det_f1, det_top = [], []
    for sd, s in scores_by_seed.items():
        det_f1.append(s >= taus[sd])
        top = np.zeros(len(s), dtype=bool)
        order = np.lexsort((np.random.default_rng(0).random(len(s)), -s))
        top[order[:k]] = True
        det_top.append(top)
    det_f1, det_top = np.mean(det_f1, axis=0), np.mean(det_top, axis=0)
    t = pd.DataFrame({"typology": typology[lab], "f1": det_f1[lab], "top": det_top[lab]})
    out = t.groupby("typology").agg(n=("f1", "size"), recall_at_f1=("f1", "mean"),
                                    recall_in_top=("top", "mean"))
    return out.sort_values("n", ascending=False)


def with_tail(df: pd.DataFrame) -> pd.DataFrame:
    """F1 at the validation F1 threshold, on the test block alone and with the tail added."""
    d = df.copy()
    tp, fp, fn = d.test_tp + d.tail_tp, d.test_fp + d.tail_fp, d.test_fn + d.tail_fn
    p, r = tp / (tp + fp).clip(lower=1), tp / (tp + fn).clip(lower=1)
    d["f1_with_tail"] = np.where(p + r > 0, 2 * p * r / (p + r).where(p + r > 0, 1), 0.0)
    return d.groupby(["variant", "family", "set"])[["f1", "f1_with_tail"]].mean()


def ap_with_tail(store_dir, variant, family, set_name, seeds, y_test, y_tail) -> float:
    vals = []
    for sd in seeds:
        sc = load_scores(store_dir, variant, family, set_name, sd)
        vals.append(average_precision_score(np.concatenate([y_test, y_tail]),
                                            np.concatenate([sc["test"], sc["tail"]])))
    return float(np.mean(vals))


In [ ]:
%%writefile tests/test_report.py
import numpy as np
import pandas as pd
import pytest
from sklearn.metrics import average_precision_score

from aml import report


def test_weighted_ap_matches_scikit_learn():
    rng = np.random.default_rng(0)
    y = (rng.random(3000) < 0.05).astype(int)
    s = np.round(rng.random(3000) + y * 0.3, 2)            # rounded: many ties
    w = rng.poisson(1.0, 3000).astype(float)
    p = report.SortedScores(y, s)
    assert p.ap() == pytest.approx(average_precision_score(y, s))
    assert p.ap(w) == pytest.approx(average_precision_score(y, s, sample_weight=w))


def test_paired_bootstrap():
    rng = np.random.default_rng(1)
    y = (rng.random(4000) < 0.05).astype(int)
    base = rng.random(4000)
    scores = {}
    for sd in range(3):
        noise = rng.normal(0, 0.05, 4000)
        scores[("m", "A", sd)] = base + noise
        scores[("m", "B", sd)] = base + noise + 0.8 * y          # clearly better
        scores[("m", "C", sd)] = base + noise + 0.8 * y          # identical to B
    out = report.paired_bootstrap(y, scores, [("m", "B", "A"), ("m", "C", "B")], n_boot=200)
    ba, cb = out.iloc[0], out.iloc[1]
    assert ba.difference > 0 and ba.interval_excludes_0
    assert cb.difference == pytest.approx(0) and not cb.interval_excludes_0


def test_friedman_nemenyi():
    t = pd.DataFrame({"x": [0.9, 0.8, 0.85, 0.7], "y": [0.5, 0.4, 0.45, 0.3], "z": [0.6, 0.65, 0.5, 0.35]})
    r = report.friedman_nemenyi(t)
    assert list(r["average_ranks"]) == ["x", "z", "y"] and r["average_ranks"]["x"] == 1.0
    assert r["critical_difference"] == pytest.approx(2.343 * np.sqrt(3 * 4 / 24))


def test_typology_recall():
    y = np.array([1, 1, 1, 0, 0, 0, 0, 0])
    typ = np.array(["cycle", "cycle", "no pattern", "x", "x", "x", "x", "x"])
    s = {0: np.array([.9, .1, .8, .2, .3, .1, .1, .1])}
    out = report.typology_recall(y, typ, s, {0: 0.5}, budget=0.25)    # top 2 rows flagged
    assert out.loc["cycle", "recall_at_f1"] == 0.5 and out.loc["no pattern", "recall_in_top"] == 1.0


In [ ]:
# Use the modules just written, then run the tests of this notebook
import importlib
for name in [n for n in list(sys.modules) if n.split(".")[0] in ("aml", "tests")]:
    del sys.modules[name]
importlib.invalidate_caches()
res = subprocess.run([sys.executable, "-m", "pytest", "-q", "-p", "no:cacheprovider", "tests/test_report.py"],
                     capture_output=True, text=True)
print(res.stdout[-1500:], res.stderr[-1500:])
assert res.returncode == 0, "tests failed"
from aml import report, sweep
from aml import model_params as MP
if DRY_RUN:
    N_BOOT = 50

In [ ]:
# Records of Notebook 03: one row per final fit (variant x family x set x seed)
df = report.load_final(STORE)
df = df[df["variant"].isin(VARIANTS)]
counts = df.groupby(["variant", "family", "set"])["seed"].count()
print(f"{len(df)} final fits; seeds per configuration: {sorted(counts.unique())}")
assert counts.min() == counts.max(), "some configurations are incomplete: finish Notebook 03 first"
first = df.groupby("variant")[["n_pos", "n"]].first()
base_rate = first["n_pos"] / first["n"]
NAMES = report.FAMILY_NAMES
SET_COLOURS = {"A": "tab:blue", "B": "tab:orange", "C": "tab:green"}
display(pd.DataFrame({"test rows (without tail)": df.groupby("variant")["n"].first(),
                      "laundering": df.groupby("variant")["n_pos"].first(),
                      "base rate per 1,000": (1000 * base_rate).round(3)}))

## 1 Overview of results

Means over five seeds, standard deviations in brackets, on the test block without the tail.
Average precision is the primary measure (Section 4.9); recall among the highest-scoring 0.1 %
of the test block is the operational measure, and precision and recall at the F1 threshold fixed
on validation describe the decision rule.

In [ ]:
# Overview: the main result tables of Chapter 5
tables = {"test_ap": ("Average precision", 3), "recall_top": ("Recall in the top 0.1 %", 3),
          "f1": ("F1 at the validation threshold", 3), "precision_f1": ("Precision at the F1 threshold", 3),
          "recall_f1": ("Recall at the F1 threshold", 3), "test_roc_auc": ("ROC-AUC (Appendix C only)", 3)}
for metric, (title, digits) in tables.items():
    t = report.mean_sd(df, metric, digits)
    t.index = pd.MultiIndex.from_tuples([(v, NAMES[f]) for v, f in t.index], names=["variant", "family"])
    print(title)
    display(t)
    save_tab(t, f"table_{metric}")

In [ ]:
# Overview: average precision by family and feature set (mean +- sd over seeds)
fig, axes = plt.subplots(1, len(VARIANTS), figsize=(7 * len(VARIANTS), 5), squeeze=False)
for ax, v in zip(axes[0], VARIANTS):
    g = df[df.variant == v].groupby(["family", "set"])["test_ap"].agg(["mean", "std"])
    m, sd = g["mean"].unstack("set").loc[report.FAMILIES], g["std"].unstack("set").loc[report.FAMILIES]
    m.index = sd.index = [NAMES[f] for f in m.index]
    m.plot.bar(yerr=sd, capsize=3, ax=ax, color=[SET_COLOURS[c] for c in m.columns], rot=20)
    ax.axhline(base_rate[v], ls=":", color="grey", label="base rate (random model)")
    ax.set_ylabel("test average precision"); ax.set_title(v); ax.legend(title="feature set")
fig.tight_layout()
save_fig(fig, "ap_by_family_and_set")
plt.show()

## 2 RQ1: model families

Following Demšar (2006), the families are ranked within each data condition (two variants × three
feature sets = six conditions, using the seed-mean test average precision), the Friedman test asks
whether the average ranks differ more than chance would allow, and the Nemenyi critical difference
marks which pairs of families are separated. With six conditions the test has little power, so
pairs closer than the critical difference are reported as not separated rather than ranked.

In [ ]:
# RQ1: ranks of the families in each condition, Friedman test and Nemenyi critical difference
rq1 = {}
for metric in ["test_ap", "recall_top"]:
    tab = df.groupby(["variant", "set", "family"])[metric].mean().unstack("family")[report.FAMILIES]
    tab.columns = [NAMES[c] for c in tab.columns]
    r = report.friedman_nemenyi(tab)
    rq1[metric] = r
    print(f"{metric}: Friedman chi2 = {r['statistic']:.2f}, p = {r['p_value']:.4f}, "
          f"N = {r['n_blocks']} conditions, k = {r['k']}; critical difference = {r['critical_difference']:.2f}")
    display(r["ranks"].assign(**{"": ""}).drop(columns="").round(1))
    display(pd.Series(r["average_ranks"], name="average rank").round(2).to_frame())
    save_tab(r["ranks"], f"rq1_ranks_{metric}")

In [ ]:
# RQ1: critical-difference diagram (average rank; a bar joins families closer than the CD)
r = rq1["test_ap"]
avg = pd.Series(r["average_ranks"]).sort_values()
cd = r["critical_difference"]
fig, ax = plt.subplots(figsize=(9, 2.8))
ax.set_xlim(0.5, len(avg) + 0.5); ax.set_ylim(-1.2, 1.6); ax.invert_xaxis()
ax.axhline(0, color="black", lw=1)
for i, (name, rank) in enumerate(avg.items()):
    y = 0.9 - 0.35 * (i % 3)
    ax.plot([rank, rank], [0, y], color="black", lw=0.8)
    ax.text(rank, y + 0.08, f"{name}\n{rank:.2f}", ha="center", va="bottom", fontsize=9)
ax.plot([avg.iloc[0], avg.iloc[0] + cd], [-0.6, -0.6], color="tab:red", lw=3)
ax.text(avg.iloc[0] + cd / 2, -1.0, f"CD = {cd:.2f}", ha="center", color="tab:red")
groups = [(a, b) for a in avg.index for b in avg.index if avg[a] < avg[b] and avg[b] - avg[a] < cd]
for k, (a, b) in enumerate(groups):
    ax.plot([avg[a], avg[b]], [-0.25 - 0.08 * k, -0.25 - 0.08 * k], color="grey", lw=2)
ax.set_yticks([]); ax.set_xlabel("average rank (1 = best), test average precision")
ax.set_title(f"Families across six conditions (Friedman p = {r['p_value']:.3f})")
for s in ["left", "right", "top"]:
    ax.spines[s].set_visible(False)
save_fig(fig, "rq1_critical_difference")
plt.show()

## 3 RQ2: feature sets

Two complementary analyses. First, the rank procedure again, now with the three sets as
treatments and the ten family × variant combinations as conditions. Second, a paired bootstrap of
the test block for each family and variant: in each replicate every test transaction receives one
Poisson(1) weight, the same for all sets and seeds, and the statistic is the seed-averaged
difference in average precision. The seeds capture variation from training; the bootstrap adds the
uncertainty of a test block with fewer than a thousand laundering transactions. This bootstrap is
an addition to Section 4.9 and should be described there.

In [ ]:
# RQ2: ranks of the sets within each family x variant, Friedman test and critical difference
tab = df.groupby(["variant", "family", "set"])["test_ap"].mean().unstack("set")[report.SETS]
r2 = report.friedman_nemenyi(tab)
print(f"Friedman chi2 = {r2['statistic']:.2f}, p = {r2['p_value']:.4f}, N = {r2['n_blocks']}, k = 3; "
      f"critical difference = {r2['critical_difference']:.2f}")
display(pd.concat([tab.round(4), r2["ranks"].add_prefix("rank ")], axis=1))
display(pd.Series(r2["average_ranks"], name="average rank").round(2).to_frame())
save_tab(r2["ranks"], "rq2_ranks")

In [ ]:
# RQ2: paired bootstrap of the differences B - A, C - B and C - A (about 10-15 minutes per variant)
boot = []
for v in VARIANTS:
    rows = report.test_rows(v)
    seeds = sorted(df[df.variant == v]["seed"].unique())
    scores = {(f, s, sd): report.load_scores(STORE, v, f, s, sd)["test"]
              for f in report.FAMILIES for s in report.SETS for sd in seeds}
    contrasts = [(f, hi, lo) for f in report.FAMILIES for hi, lo in [("B", "A"), ("C", "B"), ("C", "A")]]
    t = report.paired_bootstrap(rows["y_test"], scores, contrasts, n_boot=N_BOOT, seed=0)
    t.insert(0, "variant", v)
    boot.append(t)
    print(f"{v}: done")
boot = pd.concat(boot, ignore_index=True)
boot["family"] = boot["group"].map(NAMES)
show = boot[["variant", "family", "contrast", "difference", "ci_low", "ci_high", "interval_excludes_0"]]
display(show.round(4))
save_tab(show.set_index(["variant", "family", "contrast"]), "rq2_bootstrap")

In [ ]:
# RQ2: forest plot of the bootstrap intervals
fig, axes = plt.subplots(1, len(VARIANTS), figsize=(7 * len(VARIANTS), 6), squeeze=False, sharey=True)
for ax, v in zip(axes[0], VARIANTS):
    b = boot[(boot.variant == v) & boot.contrast.isin(["B - A", "C - B"])].reset_index(drop=True)
    ypos = np.arange(len(b))[::-1]
    colours = b.contrast.map({"B - A": "tab:orange", "C - B": "tab:green"})
    for yp, (_, row), col in zip(ypos, b.iterrows(), colours):
        ax.errorbar(row.difference, yp, xerr=[[row.difference - row.ci_low], [row.ci_high - row.difference]],
                    fmt="o", color=col, capsize=3)
    ax.axvline(0, color="black", lw=1)
    ax.set_yticks(ypos)
    ax.set_yticklabels([f"{fam}: {c}" for fam, c in zip(b.family, b.contrast)])
    ax.set_xlabel("difference in test average precision (95 % bootstrap interval)")
    ax.set_title(v)
fig.tight_layout()
save_fig(fig, "rq2_bootstrap")
plt.show()

## 4 RQ3: the two variants

Average precision depends on the base rate, which halves from HI-Small to LI-Small, so the
comparison is made on the lift over a random model (average precision divided by the base rate)
as well as on the raw values.

In [ ]:
# RQ3: raw average precision, lift over the base rate, and recall in the top 0.1 %, side by side
g = df.groupby(["variant", "family", "set"])[["test_ap", "recall_top"]].mean().reset_index()
g["lift"] = g["test_ap"] / g["variant"].map(base_rate)
rq3 = g.pivot_table(index=["family", "set"], columns="variant", values=["test_ap", "lift", "recall_top"])
rq3 = rq3.loc[report.FAMILIES]
rq3.index = pd.MultiIndex.from_tuples([(NAMES[f], s) for f, s in rq3.index], names=["family", "set"])
display(rq3.round(3))
save_tab(rq3, "rq3_variants")
order = {v: g[g.variant == v].groupby("family")["test_ap"].mean().rank(ascending=False) for v in VARIANTS}
if len(VARIANTS) == 2:
    from scipy.stats import spearmanr
    a, b = (order[v].loc[report.FAMILIES] for v in VARIANTS)
    print("Family ranks (mean AP over sets):", pd.DataFrame({v: order[v] for v in VARIANTS}).loc[report.FAMILIES].rename(index=NAMES).to_dict())
    print(f"Spearman correlation of the family ranking between variants: {spearmanr(a, b).statistic:.2f}")

## 5 Laundering typologies (Section 5.5)

For each family's Set C model, the share of each typology's test transactions that is detected at
the F1 threshold (averaged over seeds). Laundering transactions without an entry in the patterns
file appear as *no pattern*. The typology labels were never used as features.

In [ ]:
# Typologies: recall at the F1 threshold per typology and family (Set C)
for v in VARIANTS:
    rows = report.test_rows(v)
    cols = {}
    for f in report.FAMILIES:
        d = df[(df.variant == v) & (df.family == f) & (df.set == "C")]
        sc = {sd: report.load_scores(STORE, v, f, "C", sd)["test"] for sd in d.seed}
        taus = dict(zip(d.seed, d.tau_f1))
        t = report.typology_recall(rows["y_test"], rows["typology_test"], sc, taus, MP.ALERT_BUDGET)
        cols[NAMES[f]] = t["recall_at_f1"]
        n = t["n"]
    out = pd.DataFrame(cols).assign(n=n).sort_values("n", ascending=False)
    out = out[["n"] + list(cols)]
    print(v)
    display(out.round(3))
    save_tab(out, f"typology_recall_{v}")

In [ ]:
# Typologies: the best configuration by validation AP, at the F1 threshold and in the top 0.1 %
for v in VARIANTS:
    best = sweep.best_configuration(sweep.Store(STORE), v)
    rows = report.test_rows(v)
    d = df[(df.variant == v) & (df.family == best["family"]) & (df["set"] == best["set"])]
    sc = {sd: report.load_scores(STORE, v, best["family"], best["set"], sd)["test"] for sd in d.seed}
    t = report.typology_recall(rows["y_test"], rows["typology_test"], sc, dict(zip(d.seed, d.tau_f1)), MP.ALERT_BUDGET)
    print(f"{v}: {NAMES[best['family']]}, set {best['set']} (chosen on validation)")
    display(t.round(3))
    save_tab(t, f"typology_best_{v}")

## 6 Imbalance treatments (Section 5.7)

In [ ]:
# Supplement: the best HI-Small configuration without weighting, with weighting, and with a tuned threshold
sup_path = STORE / "supplement.jsonl"
if sup_path.exists() and sup_path.read_text().strip():
    sup = pd.DataFrame([json.loads(l) for l in sup_path.read_text().splitlines()])
    fam, st = sup["family"].iloc[0], sup["set"].iloc[0]
    main = df[(df.variant == "HI-Small") & (df.family == fam) & (df["set"] == st)]
    main = pd.DataFrame({"condition": "weighted, tuned threshold (main result)", "test_ap": main.test_ap,
                         "precision": main.precision_f1, "recall": main.recall_f1, "f1": main.f1,
                         "alert_rate": main.alert_rate_f1})
    s = pd.concat([sup[["condition", "test_ap", "precision", "recall", "f1", "alert_rate"]], main])
    out = s.groupby("condition")[["test_ap", "precision", "recall", "f1", "alert_rate"]].agg(["mean", "std"])
    print(f"HI-Small, {NAMES[fam]}, set {st}")
    display(out.round(4))
    save_tab(out, "imbalance_supplement")
else:
    print("no supplement records")

## 7 The tail and comparability with published results

The primary evaluation excludes the sparse tail after 10 September (Section 4.5). Published results
on AMLworld are computed on test sets that probably include those days, so the same models are
also scored with the tail added: the F1 threshold is unchanged (fixed on validation), only the rows
change. These figures serve comparison with the literature in Chapter 6, not the research questions.

In [ ]:
# Comparability: F1 and average precision with the tail added (same models, same thresholds)
f1t = report.with_tail(df)
ap_rows = []
for v in VARIANTS:
    rows = report.test_rows(v)
    for f in report.FAMILIES:
        for s in report.SETS:
            seeds = df[(df.variant == v) & (df.family == f) & (df["set"] == s)]["seed"]
            ap_rows.append({"variant": v, "family": f, "set": s,
                            "ap": df[(df.variant == v) & (df.family == f) & (df["set"] == s)]["test_ap"].mean(),
                            "ap_with_tail": report.ap_with_tail(STORE, v, f, s, seeds, rows["y_test"], rows["y_tail"])})
comp = pd.DataFrame(ap_rows).set_index(["variant", "family", "set"]).join(f1t)
comp.index = pd.MultiIndex.from_tuples([(v, NAMES[f], s) for v, f, s in comp.index], names=["variant", "family", "set"])
display(comp.round(3))
save_tab(comp, "comparability_with_tail")

## 8 Selected hyperparameters and embedded selection (Appendix A)

In [ ]:
# Appendix A: the configuration selected on validation for every family, set and variant
sel = df[df.seed == 0][["variant", "family", "set", "val_ap", "params"]].copy()
sel["family"] = sel["family"].map(NAMES)
sel["params"] = sel["params"].map(lambda p: ", ".join(f"{k}={v:.4g}" if isinstance(v, float) else f"{k}={v}"
                                                       for k, v in sorted(p.items())))
sel = sel.sort_values(["variant", "family", "set"]).set_index(["variant", "family", "set"])
display(sel)
save_tab(sel, "selected_hyperparameters")
lr = df[(df.family == "logreg") & (df.seed == 0)]
emb = pd.DataFrame({"variant": lr.variant, "set": lr.set,
                    "features": lr.extras.map(lambda e: len(e["coefficients"])),
                    "zero coefficients": lr.extras.map(lambda e: e["n_zero_coefficients"]),
                    "l1_ratio": lr.params.map(lambda p: p["l1_ratio"])}).sort_values(["variant", "set"])
print("Logistic regression: embedded selection")
display(emb.set_index(["variant", "set"]))
save_tab(emb.set_index(["variant", "set"]), "embedded_selection")

In [ ]:
# Section 4.10: computation time of the search and of the final fits, per family (hours)
search = pd.DataFrame([json.loads(l) for l in (STORE / "search.jsonl").read_text().splitlines()])
search = search[search.variant.isin(VARIANTS)]
ct = pd.DataFrame({"search": search.groupby("family")["seconds"].sum() / 3600,
                   "final fits": df.groupby("family")["seconds"].sum() / 3600})
ct["total"] = ct.sum(axis=1)
ct = ct.loc[report.FAMILIES].rename(index=NAMES)
ct.loc["total"] = ct.sum()
display(ct.round(2))
save_tab(ct, "computation_time")

## 9 Precision-recall curves (figures for Chapter 5)

In [ ]:
# PR curves: the five families on Set C, and gradient boosting across the three sets (seed 0)
for v in VARIANTS:
    y = report.test_rows(v)["y_test"]
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    for f in report.FAMILIES:
        s = report.load_scores(STORE, v, f, "C", 0)["test"]
        p, r, _ = precision_recall_curve(y, s)
        ap = df[(df.variant == v) & (df.family == f) & (df["set"] == "C") & (df.seed == 0)]["test_ap"].iloc[0]
        axes[0].plot(r, p, label=f"{NAMES[f]} (AP {ap:.3f})")
    for s_name in report.SETS:
        s = report.load_scores(STORE, v, "hgb", s_name, 0)["test"]
        p, r, _ = precision_recall_curve(y, s)
        ap = df[(df.variant == v) & (df.family == "hgb") & (df["set"] == s_name) & (df.seed == 0)]["test_ap"].iloc[0]
        axes[1].plot(r, p, color=SET_COLOURS[s_name], label=f"set {s_name} (AP {ap:.3f})")
    for ax, title in zip(axes, ["Families, set C", "Gradient boosting, sets A-C"]):
        ax.axhline(base_rate[v], ls=":", color="grey", label="base rate")
        ax.set_xlabel("recall"); ax.set_ylabel("precision"); ax.set_title(f"{v}: {title} (test, seed 0)")
        ax.legend()
    fig.tight_layout()
    save_fig(fig, f"pr_curves_{v}")
    plt.show()

## 10 Findings to record (Chapter 5)

1. **5.1–5.2 Overall and RQ1:** the overview tables; the Friedman test and average ranks; which pairs
   of families the critical difference separates.
2. **5.3 RQ2:** the set ranks and, per family and variant, whether the bootstrap interval for B − A
   and for C − B excludes zero.
3. **5.4 RQ3:** raw average precision and lift in both variants; whether the family ranking is the same.
4. **5.5 Typologies:** which typologies are detected well and which are missed, and the share of
   laundering without a pattern entry.
5. **5.6–5.7 Supplementary:** the imbalance treatments; the tail and the comparison with published results.
6. **Section 4.10 and Appendix A:** computation time, selected hyperparameters, embedded selection.

## 11 Output to send back

In [ ]:
# Output: the tables and figures written by this notebook
print("tables: ", sorted(p.name for p in TAB.glob("nb04_*.csv")))
print("figures:", sorted(p.name for p in FIG.glob("nb04_*.png")))
print("Send the results folder as a zip (the models/scores subfolder may be left out).")